# Full 50-file run on Google Colab
Clones the repo, installs `requirements.txt`, downloads the 50 FineVideo files and runs `run.py`. The Hugging Face token is read with `getpass` (not echoed, not stored).

In [ ]:
# 1. hardware + setup
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!lscpu | grep -E 'Model name|^CPU\(s\)|Thread|Core'
!free -g | head -2; df -h /content | tail -1; python3 --version
!rm -rf /content/vi && git clone -q https://github.com/MuhammadMohsinAli786/video-indexing-at-speed.git /content/vi && cd /content/vi && git log --oneline -1
!cd /content/vi && pip -q install -r requirements.txt 2>&1 | tail -3
!cd /content/vi && python prepare.py 2>&1 | grep -v WARNING
print('SETUP DONE')

In [ ]:
# 2. data (token via getpass, never printed)
import getpass, time
from huggingface_hub import snapshot_download
t = time.time()
_tok = getpass.getpass('HF token: ')
snapshot_download('HuggingFaceFV/finevideo', repo_type='dataset', allow_patterns=['data/train-000[0-4]*'],
                  local_dir='/content/finevideo', token=_tok, max_workers=16)
del _tok
import glob, os
fs = sorted(glob.glob('/content/finevideo/data/*.parquet'))
print(len(fs), 'files', round(sum(os.path.getsize(f) for f in fs) / 1e9, 2), 'GB', round(time.time() - t), 's')
print('DATA DONE')

In [ ]:
# 3. full run (timed, offline behaviour is the same as in the grader's run)
!mkdir -p /content/out && cd /content/vi && date -u && /usr/bin/time -v python run.py --input /content/finevideo/data --output /content/out/part1.jsonl --done-file /content/out/part1.jsonl.done --workers 2 > /content/out/part1.log 2>&1; date -u
!grep -v WARNING /content/out/part1.log | tail -25
print('RUN DONE')

In [ ]:
# 4. merge, compress, download
!cd /content/vi && python tools/merge_parts.py --input /content/finevideo/data --parts /content/out/part1.jsonl --out /content/out/results.jsonl.gz
!ls -la /content/out
from google.colab import files
files.download('/content/out/results.jsonl.gz')
files.download('/content/out/part1.log')
print('DOWNLOAD DONE')